In [ ]:
from application.langchain_milvus_ingest import store_chunks_to_milvus

In [ ]:
from chunking import pages_to_chunks
from pdf_with_tables import extract_pdf_content

In [8]:
import os

In [1]:
from pymilvus import connections, utility, Collection

In [ ]:
folder_path = 'assets'

In [ ]:
# for filename in os.listdir(folder_path):
#     filepath = os.path.join(folder_path, filename)
#     print(filepath)
#     pdf_pages = extract_pdf_content(filepath)
#     filename = os.path.basename(filepath)
#     all_chunks = pages_to_chunks(pdf_pages, source=filename, language='id')
    

In [ ]:
all_chunks = []
for filename in os.listdir(folder_path):
    filepath = os.path.join(folder_path, filename)
    if filepath.endswith('.pdf'):
        print(filepath)
        pdf_pages = extract_pdf_content(filepath)

        filename = os.path.basename(filepath)
        chunks = pages_to_chunks(pdf_pages, source=filename)
        all_chunks.extend(chunks)

In [ ]:
all_chunks[-50]

In [ ]:
for chunk in all_chunks:
#     if chunk['source']=='CP_Perjalanan Dinas.pdf' and chunk['element_type']=='table_row' and chunk['page']==25:
    if chunk['source']=='CP_Perjalanan Dinas.pdf' and chunk['page']==26:
        print(chunk['chunk_text'], '\n\n')

In [ ]:
for chunk in all_chunks:
    if chunk['source']=='CP-Bantuan-Pengobatan-2025.pdf':
        print(chunk['chunk_text'], '\n\n')

In [3]:
HOST = "127.0.0.1"
PORT = "19530"
COL = "company_policies"  # your collection name

# 1) connect
connections.connect(alias="default", host=HOST, port=PORT)

# 2) show collections
print("Collections:", utility.list_collections())

# 3) does collection exist?
print("Has collection:", utility.has_collection(COL))

# 4) inspect schema (if exists)
if utility.has_collection(COL):
    c = Collection(COL)
    print("Num entities:", c.num_entities)
    print("Schema fields:")
    for f in c.schema.fields:
        print("  ", f.name, f.dtype, getattr(f, "dim", None))

# 5) run a checksum query (example)
if utility.has_collection(COL):
    rows = c.query(expr='checksum in ["sha256-some"]', output_fields=["checksum","metadata"])
    print(rows)
coll = c

Collections: ['pdf_docs_test', 'milvus_langchain_test', 'company_policies']
Has collection: True
Num entities: 154
Schema fields:
   id 21 None
   embedding 101 768
   checksum 21 None
   text 21 None
   metadata 23 None


2025-09-29 13:48:09,906 [ERROR][handler]: RPC error: [query], <MilvusException: (code=503, message=no available shard leaders: channel not available[channel=by-dev-rootcoord-dml_3_461139777369885885v0])>, <Time:{'RPC start': '2025-09-29 13:48:03.858124', 'RPC error': '2025-09-29 13:48:09.906184'}> (decorators.py:140)


MilvusException: <MilvusException: (code=503, message=no available shard leaders: channel not available[channel=by-dev-rootcoord-dml_3_461139777369885885v0])>

In [ ]:
if utility.has_collection("company_policies"):
    utility.drop_collection("company_policies")
    print("deleted")

In [ ]:
skipped, inserted = store_chunks_to_milvus(
    chunks=all_chunks,
    collection_name="company_policies",
    milvus_host="127.0.0.1",
    milvus_port="19530",
    embedding_model="models/embedding-001",
    embed_batch_size=32,
    dedupe_with_milvus=True,
    dry_run=False
)
print("skipped, inserted =", skipped, inserted)


In [ ]:
# assume coll exists from previous snippet
checksum = "sha256-your-checksum-here"
rows = coll.query(expr=f'checksum in ["{checksum}"]', output_fields=["id","checksum","metadata"])
print("Query by checksum ->", rows)
# or by chunk_id if you stored it as primary id:
chunk_id = "c-12345"
rows2 = coll.query(expr=f'id in ["{chunk_id}"]', output_fields=["id","checksum","metadata"])
print("Query by id ->", rows2)


In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings  # or fallback import

C:\Users\USER\Miniconda3\envs\kpn-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
# many wrappers read env automatically; if yours needs explicit key, pass google_api_key=api_key
emb = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=api_key)

In [12]:
query_text = "Berapa biaya akomodasi untuk perjalanan dinas golongan 6 ke Sumatera?"
qvec = emb.embed_documents([query_text])[0]   # a single vector (list of floats)
print("Query vector dim:", len(qvec))

Query vector dim: 768


In [ ]:
# coll from earlier
TOP_K = 5
# search params for HNSW/COSINE index — tune ef (higher -> higher recall, slower)
search_params = {"metric_type": "COSINE", "params": {"ef": 100}}

results = coll.search(
    data=[qvec],                 # list of query vectors (can search multiple at once)
    anns_field="embedding",      # the name of your vector field
    param=search_params,
    limit=TOP_K,
    output_fields=["id", "checksum", "metadata"]
)

# results is a list (one item per query) of "hits" lists
for hits in results:
    for hit in hits:
        # printing Hit attributes - depending on pymilvus version use .id or .entity.get("id")
        try:
            entity_id = hit.id
        except Exception:
            entity_id = hit.entity.get("id") if hasattr(hit, "entity") else None
        # score/distance: with COSINE higher is more similar (closer to 1). Many APIs expose .distance or .score
        score = getattr(hit, "score", None) or getattr(hit, "distance", None)
        metadata = hit.entity.get("metadata") if hasattr(hit, "entity") else None
        print("id:", entity_id, "score:", score, "checksum:", hit.entity.get("checksum") if hasattr(hit,"entity") else None, "meta:", metadata)


In [13]:
for f in c.schema.fields:
    print(f" - {f.name} dtype={f.dtype} dim={getattr(f,'dim',None)} max_length={getattr(f,'max_length',None)}")

 - id dtype=21 dim=None max_length=128
 - embedding dtype=101 dim=768 max_length=None
 - checksum dtype=21 dim=None max_length=128
 - text dtype=21 dim=None max_length=65535
 - metadata dtype=23 dim=None max_length=None


In [16]:
# Safe expr for VARCHAR id primary key — returns rows where id is non-empty
expr = "id != ''"
rows = c.query(expr=expr, output_fields=["id", "text", "metadata"], limit=5)
print("Sample rows (up to 5):")
for r in rows:
    print(r['text'], '\n\n')

Sample rows (up to 5):
BANTUAN KACAMATA 71 ● Diberikan Kepada: ○ Hanya untuk Karyawan Tetap dan Karyawan Kontrak, setelah karyawan bekerja 72 selama 3 (tiga) bulan berturut-turut di Perusahaan. ● Yang Dapat Diklaim ke Perusahaan: 73 1. 74 Lensa dan bingkai kacamata, maksimal klaim 1 kali dalam periode 1 tahun. 2. Soft or hard contact lens sebanyak 1 pasang, maksimal klaim 1 kali dalam periode 1 tahun. 75 ● 76 Catatan Lain: 1. Cairan pembersih tidak dapat diklaim. 77 2. 


PERJALANAN DINAS – AKOMODASI PERJALANAN DINAS 


54 Apresiasi akan disesuaikan secara proporsional. CUTI APRESIASI – JUMLAH PEMBERIAN (dalam hari kerja) ● Masa Kerja < 5 tahun: ○ Non-Remote Area (Gol. 2-7): 0 hari 55 ○ Remote Area (Gol. 2-3): 2 hari 56 57 ○ Remote Area (Gol. 4-7): 4 hari ○ 58 Golongan 8-10: 9 hari ● Masa Kerja 5 - <10 tahun: 59 ○ Non-Remote Area (Gol. 2-7): 2 hari ○ 60 Remote Area (Gol. 2-3): 4 hari ○ Remote Area (Gol. 4-7): 6 hari 61 ○ Golongan 8-10: 9 hari 62 ● Masa Kerja 10 - <15 tahun: ○ Non-Remot

In [18]:
# Safe expr for VARCHAR id primary key — returns rows where id is non-empty
expr = "id != ''"
rows = c.query(expr=expr, output_fields=["id", "text", "metadata"], limit=5)
print("Sample rows (up to 5):")
for r in rows:
    print(r, '\n\n')

Sample rows (up to 5):
{'id': '03d733f9fcb84feeb304e91aad87127c', 'text': 'BANTUAN KACAMATA 71 ● Diberikan Kepada: ○ Hanya untuk Karyawan Tetap dan Karyawan Kontrak, setelah karyawan bekerja 72 selama 3 (tiga) bulan berturut-turut di Perusahaan. ● Yang Dapat Diklaim ke Perusahaan: 73 1. 74 Lensa dan bingkai kacamata, maksimal klaim 1 kali dalam periode 1 tahun. 2. Soft or hard contact lens sebanyak 1 pasang, maksimal klaim 1 kali dalam periode 1 tahun. 75 ● 76 Catatan Lain: 1. Cairan pembersih tidak dapat diklaim. 77 2.', 'metadata': {'chunk_id': '03d733f9fcb84feeb304e91aad87127c', 'source': 'CP-Bantuan-Pengobatan-2025.pdf', 'page': 5, 'element_type': 'narrative', 'table_id': None, 'row_id': None, 'list_id': None}} 


{'id': '0495abb256be4713a9768fa8d2e89f41', 'text': 'PERJALANAN DINAS – AKOMODASI PERJALANAN DINAS', 'metadata': {'chunk_id': '0495abb256be4713a9768fa8d2e89f41', 'source': 'CP_Perjalanan Dinas.pdf', 'page': 21, 'element_type': 'narrative', 'table_id': None, 'row_id': None,

In [23]:
for key, val in r.items():
    print(key)
    if type(val)==dict:
        for k, v in val.items():
            print('\t', k)

id
text
metadata
	 chunk_id
	 source
	 page
	 element_type
	 table_id
	 row_id
	 list_id


In [ ]:
for hits in results:
    for hit in hits:
        # printing Hit attributes - depending on pymilvus version use .id or .entity.get("id")
        try:
            entity_id = hit.id
        except Exception:
            entity_id = hit.entity.get("id") if hasattr(hit, "entity") else None
        # score/distance: with COSINE higher is more similar (closer to 1). Many APIs expose .distance or .score
        score = getattr(hit, "score", None) or getattr(hit, "distance", None)
        metadata = hit.entity.get("metadata") if hasattr(hit, "entity") else None
#         print("id:", entity_id, "score:", score, "checksum:", hit.entity.get("checksum") if hasattr(hit,"entity") else None, "meta:", metadata)
        print(hit, '\n')


In [ ]:
# Query all chunks from page 3
rows = coll.query(
    expr="page == 3", 
    output_fields=["id", "page", "metadata"]
)
print("Results:", rows)

In [ ]:
all_chunks[-10]